In [1]:
#!/usr/bin/env python
"""
CN radical (9e,16o)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE VERSION

Open-shell doublet radical with triple bond.
Same bond type as N2 but open-shell → α/β asymmetry.
NELEC = (5, 4): α has 4368 strings, β has 1820 strings.

States: D0, D1, D2, Q1 (3 doublets + 1 quartet)
Full space: C(16,5) × C(16,4) = 4368 × 1820 = 7,949,760 det

wf_amps[state_label] = {ref_a/b, ext_a/b, cipsi_a/b}
All on full space (0-padded).
"""

# %% Cell 1: Imports
import os, time, json, numpy as np

from pyscf import gto, scf, cc, mcscf, ao2mo, fci, lib
from pyscf.fci import selected_ci, cistring
from qiskit.circuit import QuantumCircuit
from qiskit_addon_sqd.fermion import (
    enlarge_batch_from_transitions,
    bitstring_matrix_to_ci_strs,
    recover_configurations,
    subsample,
    postselect_by_hamming_right_and_left,
    solve_fermion,
)
import ffsim
import warnings
warnings.filterwarnings("ignore")

N_THREADS = int(os.environ.get("OMP_NUM_THREADS", 24))
lib.num_threads(N_THREADS)
print(f"PySCF threads: {N_THREADS}")

# %% Cell 2: System & algorithm parameters

NORB = 16
NELEC = (5, 4)      # open-shell doublet, 9 active electrons
NCORE = 2            # C 1s + N 1s frozen

TARGET_STATES = ["D0", "D1", "D2", "Q1"]
MAX_DOUBLET = 3
MAX_QUARTET = 1

# SQD pipeline
N_SHOTS = 100000
N_SEEDS = 1
NOISE_LEVEL = 0.1
S_CORE_ITER = 5
N_BATCHES = 10
SAMPLES_PER_BATCH = 500
MERGE_TOP_K = 1
NROOTS_INT = 20

# CIPSI
CIPSI_ACUT = 3e-3
CIPSI_BROAD_EPS = 5e-4
CIPSI_EN_EPS = 1e-6
CIPSI_MAX_ITER = 10
CIPSI_EXT_ROOTS = 10
CIPSI_BROAD_MULT = 10
CIPSI_BROAD_FRAC = 0.01
NROOTS_GROW = 20
GROW_CONV_TOL = 1e-4
GROW_MAX_CYCLE = 30


# %% Cell 3: Core functions

def safe_array(x):
    return np.nan_to_num(np.asarray(x, float))

def det_strings(norb, nelec):
    na, nb = nelec
    return (np.array(cistring.make_strings(range(norb), na), np.int64),
            np.array(cistring.make_strings(range(norb), nb), np.int64))

def reverse_bits(s, norb):
    result = 0
    for i in range(norb):
        if (s >> i) & 1:
            result |= (1 << (norb - 1 - i))
    return result

def fix_ci_strs(strs, norb):
    return np.array(sorted(set(reverse_bits(int(s), norb) for s in strs)), dtype=np.int64)

def classify_spin(s2):
    if abs(s2 - 0.75) < 0.4: return "doublet"
    if abs(s2 - 3.75) < 0.5: return "quartet"
    return "other"

def kernel_safe(myci, h1, eri, norb, nelec, ci_strs, nroots, loose=False):
    try:
        if loose:
            myci.conv_tol = GROW_CONV_TOL; myci.max_cycle = GROW_MAX_CYCLE
        else:
            myci.conv_tol = 1e-10; myci.max_cycle = 200
        e, c = selected_ci.kernel_fixed_space(
            myci, h1, eri, norb, nelec, ci_strs=ci_strs, nroots=nroots)
        if np.isscalar(e): return [float(e)], [c]
        return [float(x) for x in e], list(c)
    except:
        return [], []

def get_alpha_coeffs(c_full, strsa, basis, ci_strs):
    c = np.asarray(c_full)
    coeffs = np.zeros(len(basis))
    if c.ndim == 2:
        sm = {int(s): i for i, s in enumerate(ci_strs[0])}
        for bi, bs in enumerate(basis):
            ia = sm.get(int(bs))
            if ia is not None and ia < c.shape[0]:
                coeffs[bi] = np.max(np.abs(c[ia, :]))
    return coeffs

def get_beta_coeffs(c_full, strsb, basis_b, ci_strs):
    c = np.asarray(c_full)
    coeffs = np.zeros(len(basis_b))
    if c.ndim == 2:
        sm = {int(s): i for i, s in enumerate(ci_strs[1])}
        for bi, bs in enumerate(basis_b):
            ib = sm.get(int(bs))
            if ib is not None and ib < c.shape[1]:
                coeffs[bi] = np.max(np.abs(c[:, ib]))
    return coeffs

def compute_state_marginals(ci_vec, sub_a, sub_b, full_a_map, full_b_map,
                             n_alpha_full, n_beta_full):
    c = np.asarray(ci_vec, float)
    if c.ndim != 2: return None, None
    pa_sub = np.sum(c**2, axis=1)
    pb_sub = np.sum(c**2, axis=0)
    pa_full = np.zeros(n_alpha_full)
    pb_full = np.zeros(n_beta_full)
    for i_sub in range(min(len(sub_a), len(pa_sub))):
        i_full = full_a_map.get(int(sub_a[i_sub]))
        if i_full is not None: pa_full[i_full] = pa_sub[i_sub]
    for i_sub in range(min(len(sub_b), len(pb_sub))):
        i_full = full_b_map.get(int(sub_b[i_sub]))
        if i_full is not None: pb_full[i_full] = pb_sub[i_sub]
    return pa_full, pb_full

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False

if HAS_NUMBA:
    @njit(cache=True)
    def _in_sorted(val, arr):
        """Binary search in sorted int64 array."""
        lo, hi = 0, len(arr) - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if arr[mid] == val: return True
            elif arr[mid] < val: lo = mid + 1
            else: hi = mid - 1
        return False

    @njit(cache=True)
    def _compute_diag_jit(s, norb, h1, eri):
        """JIT diagonal H element."""
        occ = np.empty(norb, dtype=np.int64); nocc = 0
        for i in range(norb):
            if (s >> i) & 1:
                occ[nocc] = i; nocc += 1
        e = 0.0
        for ii in range(nocc):
            i = occ[ii]; e += h1[i, i]
        for ii in range(nocc):
            i = occ[ii]
            for jj in range(ii+1, nocc):
                j = occ[jj]; e += eri[i, i, j, j] - eri[i, j, j, i]
        return e

    @njit(cache=True)
    def _en_couplings_one_config(s, ck, norb, h1, eri, bs_sorted):
        """JIT: compute S/D couplings from one source config.
        Returns (candidate_configs, coupling_values) arrays."""
        occ = np.empty(norb, dtype=np.int64); nocc = 0
        vir = np.empty(norb, dtype=np.int64); nvir = 0
        for i in range(norb):
            if (s >> i) & 1: occ[nocc] = i; nocc += 1
            else: vir[nvir] = i; nvir += 1

        max_n = nocc * nvir + (nocc * (nocc-1) // 2) * (nvir * (nvir-1) // 2)
        cands = np.empty(max_n, dtype=np.int64)
        coups = np.empty(max_n, dtype=np.float64)
        n = 0

        # Singles
        for ii in range(nocc):
            i = occ[ii]
            for aa in range(nvir):
                a = vir[aa]
                ns = s ^ (1 << i) ^ (1 << a)
                if _in_sorted(ns, bs_sorted): continue
                v = h1[a, i]
                for jj in range(nocc):
                    j_occ = occ[jj]
                    v += eri[a, j_occ, i, j_occ] - eri[a, j_occ, j_occ, i]
                cands[n] = ns; coups[n] = v * ck; n += 1

        # Doubles
        for ii in range(nocc):
            i = occ[ii]
            for jj in range(ii+1, nocc):
                j = occ[jj]
                for aa in range(nvir):
                    a = vir[aa]
                    for bb in range(aa+1, nvir):
                        b = vir[bb]
                        ns = s ^ (1 << i) ^ (1 << j) ^ (1 << a) ^ (1 << b)
                        if _in_sorted(ns, bs_sorted): continue
                        v = eri[a, i, b, j] - eri[a, j, b, i]
                        cands[n] = ns; coups[n] = v * ck; n += 1
        return cands[:n], coups[:n]

    @njit(cache=True)
    def _broad_one_config(s, norb, eri, eps, bs_sorted):
        """JIT: broad mode S/D from one config. Singles always added, doubles eps-filtered."""
        occ = np.empty(norb, dtype=np.int64); nocc = 0
        vir = np.empty(norb, dtype=np.int64); nvir = 0
        for i in range(norb):
            if (s >> i) & 1: occ[nocc] = i; nocc += 1
            else: vir[nvir] = i; nvir += 1

        max_n = nocc * nvir + (nocc * (nocc-1) // 2) * (nvir * (nvir-1) // 2)
        cands = np.empty(max_n, dtype=np.int64)
        n = 0

        for ii in range(nocc):
            i = occ[ii]
            for aa in range(nvir):
                a = vir[aa]
                ns = s ^ (1 << i) ^ (1 << a)
                if not _in_sorted(ns, bs_sorted):
                    cands[n] = ns; n += 1

        for ii in range(nocc):
            i = occ[ii]
            for jj in range(ii+1, nocc):
                j = occ[jj]
                for aa in range(nvir):
                    a = vir[aa]
                    for bb in range(aa+1, nvir):
                        b = vir[bb]
                        ns = s ^ (1 << i) ^ (1 << j) ^ (1 << a) ^ (1 << b)
                        if _in_sorted(ns, bs_sorted): continue
                        if eps > 0:
                            v = abs(eri[a, i, b, j] - eri[a, j, b, i])
                            if v < eps: continue
                        cands[n] = ns; n += 1
        return cands[:n]

    @njit(cache=True)
    def _pairwise_one_config(s, ck, norb, h1, eri, eps, bs_sorted):
        """JIT: standard pair-wise |H_ij * c_j| > eps from one config."""
        occ = np.empty(norb, dtype=np.int64); nocc = 0
        vir = np.empty(norb, dtype=np.int64); nvir = 0
        for i in range(norb):
            if (s >> i) & 1: occ[nocc] = i; nocc += 1
            else: vir[nvir] = i; nvir += 1

        max_n = nocc * nvir + (nocc * (nocc-1) // 2) * (nvir * (nvir-1) // 2)
        cands = np.empty(max_n, dtype=np.int64)
        n = 0

        for ii in range(nocc):
            i = occ[ii]
            for aa in range(nvir):
                a = vir[aa]
                ns = s ^ (1 << i) ^ (1 << a)
                if _in_sorted(ns, bs_sorted): continue
                v = h1[a, i]
                for jj in range(nocc):
                    j = occ[jj]
                    v += eri[a, j, i, j] - eri[a, j, j, i]
                if abs(v) * ck < eps: continue
                cands[n] = ns; n += 1

        for ii in range(nocc):
            i = occ[ii]
            for jj in range(ii+1, nocc):
                j = occ[jj]
                for aa in range(nvir):
                    a = vir[aa]
                    for bb in range(aa+1, nvir):
                        b = vir[bb]
                        ns = s ^ (1 << i) ^ (1 << j) ^ (1 << a) ^ (1 << b)
                        if _in_sorted(ns, bs_sorted): continue
                        v = abs(eri[a, i, b, j] - eri[a, j, b, i])
                        if v * ck < eps: continue
                        cands[n] = ns; n += 1
        return cands[:n]

    print(f"Numba JIT: enabled (4 functions compiled on first call)")

else:
    print("Numba: not available, using pure Python (slower)")


def _compute_diag(s, norb, h1, eri):
    if HAS_NUMBA:
        return _compute_diag_jit(s, norb, h1, eri)
    occ = [i for i in range(norb) if (s >> i) & 1]
    e = 0.0
    for i in occ:
        e += h1[i, i]
    for ii, i in enumerate(occ):
        for j in occ[ii+1:]:
            e += eri[i, i, j, j] - eri[i, j, j, i]
    return e

def cipsi_extend(basis, ci_coeffs, norb, ne, acut, maxd, h1, eri, eps,
              use_coeffs=True, broad_frac=0.01, en_mode=False, e0=0.0):
    bs = set(int(x) for x in basis)
    bs_sorted = np.array(sorted(bs), dtype=np.int64)  # for JIT binary search

    if use_coeffs:
        if isinstance(ci_coeffs, list):
            imp_ref = np.abs(ci_coeffs[0]) >= acut
            if imp_ref.sum() < 5:
                imp_ref[np.argsort(-np.abs(ci_coeffs[0]))[:5]] = True
        else:
            imp = np.abs(ci_coeffs) >= acut
            if imp.sum() < 5:
                imp[np.argsort(-np.abs(ci_coeffs))[:5]] = True
    else:
        cmax = np.max(np.abs(ci_coeffs)) if len(ci_coeffs) > 0 else 0
        threshold = cmax * broad_frac
        imp = np.abs(ci_coeffs) >= threshold
        if imp.sum() < 5:
            imp[np.argsort(-np.abs(ci_coeffs))[:min(5, len(basis))]] = True

    # === BROAD MODE ===
    if not use_coeffs:
        imp_idx = np.where(imp)[0]
        cands = set()
        if HAS_NUMBA:
            for k in imp_idx:
                new_configs = _broad_one_config(int(basis[k]), norb, eri, eps, bs_sorted)
                for ci in range(len(new_configs)):
                    cands.add(int(new_configs[ci]))
        else:
            for k in imp_idx:
                s = int(basis[k])
                occ = [i for i in range(norb) if (s >> i) & 1]
                vir = [i for i in range(norb) if not ((s >> i) & 1)]
                for i in occ:
                    for a in vir:
                        ns = s ^ (1 << i) ^ (1 << a)
                        if ns not in bs: cands.add(ns)
                for ii, i in enumerate(occ):
                    for j in occ[ii+1:]:
                        for aa, a in enumerate(vir):
                            for b in vir[aa+1:]:
                                ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                                if ns in bs: continue
                                if eps > 0:
                                    v = abs(eri[a,i,b,j] - eri[a,j,b,i])
                                    if v < eps: continue
                                cands.add(ns)
        r = np.array(sorted(bs | cands), np.int64)
        return r[:maxd] if maxd and len(r) > maxd else r

    # === EN MODE ===
    if en_mode:
        if isinstance(e0, (list, np.ndarray)):
            e0_list = list(e0); coeffs_list = list(ci_coeffs)
        else:
            e0_list = [e0]; coeffs_list = [ci_coeffs]

        all_couplings = []
        for ri, (e0_r, coeffs_r) in enumerate(zip(e0_list, coeffs_list)):
            imp_r = np.abs(coeffs_r) >= acut
            if imp_r.sum() < 5:
                imp_r[np.argsort(-np.abs(coeffs_r))[:5]] = True
            imp_idx_r = np.where(imp_r)[0]

            coupling = {}
            if HAS_NUMBA:
                for k in imp_idx_r:
                    s = int(basis[k]); ck = float(coeffs_r[k])
                    new_c, new_v = _en_couplings_one_config(s, ck, norb, h1, eri, bs_sorted)
                    for ci in range(len(new_c)):
                        ns = int(new_c[ci])
                        coupling[ns] = coupling.get(ns, 0.0) + new_v[ci]
            else:
                from collections import defaultdict
                coupling = defaultdict(float)
                for k in imp_idx_r:
                    s = int(basis[k]); ck = float(coeffs_r[k])
                    occ = [i for i in range(norb) if (s >> i) & 1]
                    vir = [i for i in range(norb) if not ((s >> i) & 1)]
                    for i in occ:
                        for a in vir:
                            ns = s ^ (1 << i) ^ (1 << a)
                            if ns in bs: continue
                            v = h1[a, i]
                            for j_occ in occ:
                                v += eri[a, j_occ, i, j_occ] - eri[a, j_occ, j_occ, i]
                            coupling[ns] += v * ck
                    for ii, i in enumerate(occ):
                        for j in occ[ii+1:]:
                            for aa, a in enumerate(vir):
                                for b in vir[aa+1:]:
                                    ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                                    if ns in bs: continue
                                    v = eri[a,i,b,j] - eri[a,j,b,i]
                                    coupling[ns] += v * ck
            all_couplings.append((coupling, e0_r))

        all_candidates = set()
        for coupling, _ in all_couplings:
            all_candidates |= set(coupling.keys())
        cands = set()
        for ns in all_candidates:
            h_ii = _compute_diag(ns, norb, h1, eri)
            for coupling, e0_r in all_couplings:
                coup = coupling.get(ns, 0.0)
                if abs(coup) < 1e-15: continue
                denom = abs(e0_r - h_ii)
                if denom < 1e-12: denom = 1e-12
                delta_e = coup * coup / denom
                if delta_e > eps:
                    cands.add(ns); break
        r = np.array(sorted(bs | cands), np.int64)
        return r[:maxd] if maxd and len(r) > maxd else r

    # === STANDARD PAIR-WISE MODE ===
    imp_idx = np.where(imp)[0]
    cands = set()
    if HAS_NUMBA:
        for k in imp_idx:
            new_configs = _pairwise_one_config(int(basis[k]), abs(float(ci_coeffs[k])),
                                                norb, h1, eri, eps, bs_sorted)
            for ci in range(len(new_configs)):
                cands.add(int(new_configs[ci]))
    else:
        for k in imp_idx:
            s = int(basis[k]); ck = abs(ci_coeffs[k])
            occ = [i for i in range(norb) if (s >> i) & 1]
            vir = [i for i in range(norb) if not ((s >> i) & 1)]
            for i in occ:
                for a in vir:
                    ns = s ^ (1 << i) ^ (1 << a)
                    if ns in bs: continue
                    v = h1[a, i]
                    for j in occ:
                        v += eri[a, j, i, j] - eri[a, j, j, i]
                    if abs(v) * ck < eps: continue
                    cands.add(ns)
            for ii, i in enumerate(occ):
                for j in occ[ii+1:]:
                    for aa, a in enumerate(vir):
                        for b in vir[aa+1:]:
                            ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                            if ns in bs: continue
                            v = abs(eri[a,i,b,j] - eri[a,j,b,i])
                            if v * ck < eps: continue
                            cands.add(ns)
    r = np.array(sorted(bs | cands), np.int64)
    return r[:maxd] if maxd and len(r) > maxd else r

def build_sd_transitions(norb, nelec):
    na, nb = nelec; ops = []
    for i in range(na):
        for a in range(na, norb):
            op = np.full(2*norb, 'I', dtype='U1'); op[norb+a]='+'; op[norb+i]='-'; ops.append(op)
    for i in range(na):
        for j in range(i+1, na):
            for a in range(na, norb):
                for b in range(a+1, norb):
                    op = np.full(2*norb, 'I', dtype='U1'); op[norb+a]='+'; op[norb+b]='+'; op[norb+i]='-'; op[norb+j]='-'; ops.append(op)
    for i in range(nb):
        for a in range(nb, norb):
            op = np.full(2*norb, 'I', dtype='U1'); op[a]='+'; op[i]='-'; ops.append(op)
    for i in range(nb):
        for j in range(i+1, nb):
            for a in range(nb, norb):
                for b in range(a+1, norb):
                    op = np.full(2*norb, 'I', dtype='U1'); op[a]='+'; op[b]='+'; op[i]='-'; op[j]='-'; ops.append(op)
    return np.array(ops)

def sample_ucj_bsm(norb, nelec, t1, t2, n_shots, noise_level=0.1, seed=42):
    rng = np.random.default_rng(seed)
    na, nb = nelec
    t2a, t1a = safe_array(t2), safe_array(t1)
    strsa, strsb = det_strings(norb, nelec); n_b = len(strsb)
    probs_merged = None; n_merged = 0
    for ts in [0.5, 1.0, 1.5, 2.0]:
        try:
            qc = QuantumCircuit(2*norb)
            qc.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), list(range(2*norb)))
            qc.barrier()
            ucj = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
                t2=(t2a*ts, t2a*ts, t2a*ts), t1=(t1a*ts, t1a*ts), n_reps=(8, 8))
            qc.append(ffsim.qiskit.UCJOpSpinUnbalancedJW(ucj), list(range(2*norb)))
            vec = ffsim.qiskit.final_state_vector(qc, norb=norb, nelec=nelec)
            p = np.abs(np.asarray(vec, np.complex128))**2
            if probs_merged is None: probs_merged = p.copy()
            else: probs_merged += p
            n_merged += 1
        except: pass
    if probs_merged is not None and n_merged > 0:
        probs_best = probs_merged / n_merged
    else:
        probs_best = np.zeros(len(strsa)*n_b); probs_best[0] = 1.0
    probs_noisy = (1-noise_level)*probs_best + noise_level/len(probs_best)
    probs_noisy /= probs_noisy.sum()
    indices = rng.choice(len(probs_noisy), size=n_shots, p=probs_noisy)
    bsm = np.zeros((n_shots, 2*norb), dtype=bool); shot_probs = np.zeros(n_shots)
    for k, idx in enumerate(indices):
        ia, ib = idx//n_b, idx%n_b; a, b = int(strsa[ia]), int(strsb[ib])
        for i in range(norb):
            if (b >> i) & 1: bsm[k, i] = True
            if (a >> i) & 1: bsm[k, norb+i] = True
        shot_probs[k] = probs_noisy[idx]
    shot_probs /= shot_probs.sum()
    return bsm, shot_probs

def label_states_fci(solver, e_list, c_list, norb, nelec, e_core):
    labeled, s2_dict, idx_map = {}, {}, {}
    d_cnt, q_cnt = 0, 0
    for k in range(len(e_list)):
        s2, _ = solver.spin_square(c_list[k], norb, nelec)
        s2 = float(s2); tag = classify_spin(s2)
        if tag == "doublet" and d_cnt < MAX_DOUBLET:
            lb = f"D{d_cnt}"; d_cnt += 1
        elif tag == "quartet" and q_cnt < MAX_QUARTET:
            lb = f"Q{q_cnt+1}"; q_cnt += 1
        else: continue
        labeled[lb] = float(e_core + e_list[k]); s2_dict[lb] = s2; idx_map[lb] = k
        if d_cnt >= MAX_DOUBLET and q_cnt >= MAX_QUARTET: break
    return labeled, s2_dict, idx_map

def label_states_sci(myci, e_list, c_list, norb, nelec, e_core):
    labeled, s2_dict, idx_map = {}, {}, {}
    d_cnt, q_cnt = 0, 0
    for k in range(len(e_list)):
        try: s2 = float(selected_ci.spin_square(c_list[k], norb, nelec)[0])
        except: s2 = -1.0
        tag = classify_spin(s2)
        if tag == "doublet" and d_cnt < MAX_DOUBLET:
            lb = f"D{d_cnt}"; d_cnt += 1
        elif tag == "quartet" and q_cnt < MAX_QUARTET:
            lb = f"Q{q_cnt+1}"; q_cnt += 1
        else: continue
        labeled[lb] = float(e_core + e_list[k]); s2_dict[lb] = s2; idx_map[lb] = k
        if d_cnt >= MAX_DOUBLET and q_cnt >= MAX_QUARTET: break
    return labeled, s2_dict, idx_map


# %% Cell 4: Build CN molecule

def build_cn(R):
    """Build CN radical at bond distance R."""
    mol = gto.M(atom=f"C 0 0 0; N 0 0 {R}",
                basis="cc-pvdz", charge=0, spin=1,
                unit="Angstrom", verbose=0, max_memory=8000)
    mf = scf.ROHF(mol).run(conv_tol=1e-12)
    na, nb = NELEC
    mc = mcscf.CASCI(mf, ncas=NORB, nelecas=NELEC)
    mc.ncore = NCORE; mc.mo_coeff = mf.mo_coeff
    h1, e_core = mc.get_h1eff()
    eri = ao2mo.restore(1, mc.get_h2eff(), NORB)

    nocc_min, nvir = min(na, nb), NORB - max(na, nb)
    try:
        mycc = cc.CCSD(mf, frozen=NCORE)
        mycc.conv_tol = 1e-10; mycc.max_cycle = 100; mycc.kernel()
        t1_raw, t2_raw = mycc.t1, mycc.t2
        if isinstance(t1_raw, (tuple, list)):
            t1 = safe_array(t1_raw[0][:nocc_min, :nvir])
        else:
            t1 = safe_array(t1_raw[:nocc_min, :nvir])
        if isinstance(t2_raw, (tuple, list)):
            t2 = safe_array(t2_raw[0][:nocc_min, :nocc_min, :nvir, :nvir])
        else:
            t2 = safe_array(t2_raw[:nocc_min, :nocc_min, :nvir, :nvir])
    except Exception as e:
        print(f"  CCSD fallback: {e}")
        rng = np.random.default_rng(42)
        t1 = rng.normal(0, 0.03, (nocc_min, nvir))
        t2 = rng.normal(0, 0.01, (nocc_min, nocc_min, nvir, nvir))
        t2 = t2 - t2.transpose(1, 0, 2, 3)
    return h1, eri, float(e_core), t1, t2


# %% Cell 5: Compute single geometry

def compute_point(R, REF_NROOTS=20):
    print(f"\n  R = {R:.2f} A", end="", flush=True)
    t0_all = time.time()
    na, nb = NELEC

    h1, eri, e_core, t1, t2 = build_cn(R)
    strsa, strsb = det_strings(NORB, NELEC)
    n_alpha_full, n_beta_full = len(strsa), len(strsb)
    full_a_map = {int(s): i for i, s in enumerate(strsa)}
    full_b_map = {int(s): i for i, s in enumerate(strsb)}

    # --- Reference ---
    t0 = time.time()
    solver = fci.direct_spin1.FCI()
    solver.conv_tol = 1e-10; solver.max_cycle = 200; solver.max_memory = 8000
    e_fci, c_fci = solver.kernel(h1, eri, NORB, NELEC, nroots=REF_NROOTS)
    if np.isscalar(e_fci):
        e_fci, c_fci = [float(e_fci)], [c_fci]
    fci_labeled, fci_s2, fci_idx_map = label_states_fci(
        solver, e_fci, c_fci, NORB, NELEC, e_core)
    dt_fci = time.time() - t0

    # === SQD Pipeline ===
    t0 = time.time()
    bsm_parts, probs_parts = [], []
    for s_idx in range(N_SEEDS):
        bsm_s, probs_s = sample_ucj_bsm(
            NORB, NELEC, t1, t2, N_SHOTS, NOISE_LEVEL, seed=42+s_idx*1000)
        bsm_parts.append(bsm_s); probs_parts.append(probs_s)
    bsm_raw = np.vstack(bsm_parts)
    probs_raw = np.concatenate(probs_parts); probs_raw /= probs_raw.sum()

    bsm_ps, probs_ps = postselect_by_hamming_right_and_left(
        bsm_raw, probs_raw, hamming_right=na, hamming_left=nb)
    for it in range(S_CORE_ITER):
        energy_sqd, sci_state, avg_occs, spin_sq = solve_fermion(
            bsm_ps, hcore=h1, eri=eri, open_shell=True)
        bsm_ps, probs_ps = recover_configurations(
            bsm_ps, probs_ps, avg_occs, na, nb, rand_seed=42+it)

    spb = min(SAMPLES_PER_BATCH, len(bsm_ps))
    batches = subsample(bsm_ps, probs_ps, spb, N_BATCHES, rand_seed=42)
    batch_energies = []
    for i, batch in enumerate(batches):
        e_b, _, _, _ = solve_fermion(batch, hcore=h1, eri=eri, open_shell=True)
        batch_energies.append((e_b, i))
    batch_energies.sort()
    merged_rows = []
    for _, idx in batch_energies[:MERGE_TOP_K]:
        merged_rows.append(batches[idx])
    merged_batch = np.vstack(merged_rows)
    _, unique_idx = np.unique(merged_batch, axis=0, return_index=True)
    merged_batch = merged_batch[np.sort(unique_idx)]

    sqd_a, sqd_b = bitstring_matrix_to_ci_strs(merged_batch, open_shell=True)
    sqd_a = fix_ci_strs(sqd_a, NORB); sqd_b = fix_ci_strs(sqd_b, NORB)
    dt_sqd = time.time() - t0

    # === Ext-SQD ===
    t0 = time.time()
    ops = build_sd_transitions(NORB, NELEC)
    ext_bsm = enlarge_batch_from_transitions(merged_batch, ops)
    ext_a, ext_b = bitstring_matrix_to_ci_strs(ext_bsm, open_shell=True)
    ext_a = fix_ci_strs(ext_a, NORB); ext_b = fix_ci_strs(ext_b, NORB)

    myci = selected_ci.SelectedCI()
    myci.conv_tol = 1e-10; myci.max_cycle = 200; myci.max_memory = 8000
    ci_ext = (ext_a, ext_b)
    nr_ext = min(NROOTS_INT, len(ext_a) * len(ext_b))
    el_ext, cl_ext = kernel_safe(myci, h1, eri, NORB, NELEC, ci_ext, nr_ext)
    ext_labeled, ext_s2, ext_idx_map = label_states_sci(
        myci, el_ext, cl_ext, NORB, NELEC, e_core)
    dt_ext = time.time() - t0

    # === CIPSI: seed → broad → EN ===
    t0 = time.time()
    seed_a = np.array(sorted(set(int(x) for x in sqd_a)), np.int64)
    seed_b = np.array(sorted(set(int(x) for x in sqd_b)), np.int64)
    basis_a, basis_b = seed_a.copy(), seed_b.copy()
    cipsi_convergence = []

    ci_seed = (seed_a, seed_b)
    e_seed, c_seed = kernel_safe(myci, h1, eri, NORB, NELEC, ci_seed,
                                  min(NROOTS_GROW, len(seed_a)*len(seed_b)), loose=True)

    all_a = set(int(x) for x in seed_a)
    all_b = set(int(x) for x in seed_b)
    if e_seed:
        for cv in c_seed[:min(CIPSI_EXT_ROOTS, len(c_seed))]:
            sca = get_alpha_coeffs(cv, strsa, seed_a, ci_seed)
            ext2_a = cipsi_extend(seed_a, sca, NORB, na, CIPSI_ACUT, 0, h1, eri,
                               CIPSI_BROAD_EPS*CIPSI_BROAD_MULT, use_coeffs=False, broad_frac=CIPSI_BROAD_FRAC)
            all_a |= set(int(x) for x in ext2_a)
            scb = get_beta_coeffs(cv, strsb, seed_b, ci_seed)
            ext2_b = cipsi_extend(seed_b, scb, NORB, nb, CIPSI_ACUT, 0, h1, eri,
                               CIPSI_BROAD_EPS*CIPSI_BROAD_MULT, use_coeffs=False, broad_frac=CIPSI_BROAD_FRAC)
            all_b |= set(int(x) for x in ext2_b)
    basis_a = np.array(sorted(all_a), np.int64)
    basis_b = np.array(sorted(all_b), np.int64)
    cipsi_convergence.append({"iter": 0, "D_a": len(basis_a), "D_b": len(basis_b), "mode": "seed-broad"})

    d_broad_a, d_broad_b = len(basis_a), len(basis_b)
    ci_broad = (basis_a, basis_b)
    nr_broad = min(NROOTS_INT, len(basis_a)*len(basis_b))
    el_broad, cl_broad = kernel_safe(myci, h1, eri, NORB, NELEC, ci_broad, nr_broad)
    broad_labeled, broad_s2, _ = label_states_sci(myci, el_broad, cl_broad, NORB, NELEC, e_core)
    print(f"\n    [BROAD] D=({len(basis_a)},{len(basis_b)})")
    for lb in TARGET_STATES:
        eb, er = broad_labeled.get(lb), fci_labeled.get(lb)
        if eb is not None and er is not None:
            print(f"      {lb}: ΔE={((eb-er)*1000):+.4f} mHa")

    for it in range(1, CIPSI_MAX_ITER):
        ci_s2 = (basis_a, basis_b)
        nr2 = min(NROOTS_GROW, len(basis_a)*len(basis_b))
        e2, c2 = kernel_safe(myci, h1, eri, NORB, NELEC, ci_s2, nr2, loose=True)
        if not e2: break

        old_a, old_b = len(basis_a), len(basis_b)
        all_a = set(int(x) for x in basis_a)
        all_b = set(int(x) for x in basis_b)
        n_roots_use = min(CIPSI_EXT_ROOTS, len(c2))
        e0_list = [float(e2[ri]) for ri in range(n_roots_use)]

        ac_list = [get_alpha_coeffs(c2[ri], strsa, basis_a, ci_s2) for ri in range(n_roots_use)]
        ext2_a = cipsi_extend(basis_a, ac_list, NORB, na, CIPSI_ACUT, 0, h1, eri,
                           CIPSI_EN_EPS, use_coeffs=True, broad_frac=CIPSI_BROAD_FRAC, en_mode=True, e0=e0_list)
        all_a |= set(int(x) for x in ext2_a)

        bc_list = [get_beta_coeffs(c2[ri], strsb, basis_b, ci_s2) for ri in range(n_roots_use)]
        ext2_b = cipsi_extend(basis_b, bc_list, NORB, nb, CIPSI_ACUT, 0, h1, eri,
                           CIPSI_EN_EPS, use_coeffs=True, broad_frac=CIPSI_BROAD_FRAC, en_mode=True, e0=e0_list)
        all_b |= set(int(x) for x in ext2_b)

        # Broad supplement on top configs
        max_coeff_a = np.zeros(len(basis_a))
        max_coeff_b = np.zeros(len(basis_b))
        for ri in range(n_roots_use):
            max_coeff_a = np.maximum(max_coeff_a, np.abs(get_alpha_coeffs(c2[ri], strsa, basis_a, ci_s2)))
            max_coeff_b = np.maximum(max_coeff_b, np.abs(get_beta_coeffs(c2[ri], strsb, basis_b, ci_s2)))

        for top_basis, top_coeffs, all_set in [(basis_a, max_coeff_a, all_a), (basis_b, max_coeff_b, all_b)]:
            top_n = max(10, int(len(top_basis)*CIPSI_BROAD_FRAC))
            top_idx = np.argsort(-top_coeffs)[:top_n]
            for s in top_basis[top_idx]:
                s_int = int(s)
                occ = [i for i in range(NORB) if (s_int >> i) & 1]
                vir = [i for i in range(NORB) if not ((s_int >> i) & 1)]
                for i in occ:
                    for a in vir:
                        all_set.add(s_int ^ (1 << i) ^ (1 << a))

        basis_a = np.array(sorted(all_a), np.int64)
        basis_b = np.array(sorted(all_b), np.int64)
        cipsi_convergence.append({"iter": it, "D_a": len(basis_a), "D_b": len(basis_b), "mode": "EN+broad"})
        if len(basis_a) == old_a and len(basis_b) == old_b: break

    ci_cipsi = (basis_a, basis_b)
    nr_cipsi = min(NROOTS_INT, len(basis_a)*len(basis_b))
    el_cipsi, cl_cipsi = kernel_safe(myci, h1, eri, NORB, NELEC, ci_cipsi, nr_cipsi)
    cipsi_labeled, cipsi_s2, cipsi_idx_map = label_states_sci(
        myci, el_cipsi, cl_cipsi, NORB, NELEC, e_core)
    dt_cipsi = time.time() - t0
    cipsi_convergence.append({"iter": "final", "D_a": len(basis_a), "D_b": len(basis_b),
                              "mode": "final", "energies": cipsi_labeled, "s2": cipsi_s2})

    print(f"    [EN] Broad D=({d_broad_a},{d_broad_b}) → Final D=({len(basis_a)},{len(basis_b)})")
    for lb in TARGET_STATES:
        eb, ef, er = broad_labeled.get(lb), cipsi_labeled.get(lb), fci_labeled.get(lb)
        if eb is not None and ef is not None and er is not None:
            print(f"      {lb}: broad={((eb-er)*1000):+.4f} → final={((ef-er)*1000):+.4f} mHa")

    # === Natural orbital occupation (GS) ===
    nat_occ = {}
    try:
        if len(c_fci) > 0:
            rdm1_ref = fci.direct_spin1.make_rdm1(c_fci[0], NORB, NELEC)
            nat_occ["ref"] = np.sort(np.linalg.eigvalsh(rdm1_ref))[::-1].tolist()
        if len(cl_ext) > 0:
            myci._strs = ci_ext
            rdm1_ext = myci.make_rdm1(cl_ext[0], NORB, NELEC)
            nat_occ["ext"] = np.sort(np.linalg.eigvalsh(rdm1_ext))[::-1].tolist()
        if len(cl_cipsi) > 0:
            myci._strs = ci_cipsi
            rdm1_cipsi = myci.make_rdm1(cl_cipsi[0], NORB, NELEC)
            nat_occ["cipsi"] = np.sort(np.linalg.eigvalsh(rdm1_cipsi))[::-1].tolist()
    except Exception as e:
        print(f"\n    [NO] Failed: {e}")

    # === Diagnostics ===
    ext_set_a = set(int(x) for x in ext_a)
    cipsi_set_a = set(int(x) for x in basis_a)
    missing_a = sorted(ext_set_a - cipsi_set_a)
    diag = {"n_missing_a": len(missing_a), "n_ext_a": len(ext_set_a), "n_cipsi_a": len(cipsi_set_a),
            "n_ext_b": len(ext_b), "n_cipsi_b": len(basis_b)}

    # WF analysis (GS)
    wf_analysis = {}
    cr_ext = None; ext_str_map_a = {}
    if len(cl_ext) > 0:
        cr_ext = np.asarray(cl_ext[0], float)
        if cr_ext.ndim == 2:
            ext_str_map_a = {int(s): i for i, s in enumerate(ext_a)}
            alpha_weights = {}
            for ia, s in enumerate(ext_a):
                if ia < cr_ext.shape[0]:
                    alpha_weights[int(s)] = float(np.sum(cr_ext[ia, :]**2))
            total_weight = sum(alpha_weights.values())
            cipsi_weight = sum(alpha_weights.get(s, 0) for s in cipsi_set_a)
            ext_only_s = ext_set_a - cipsi_set_a
            ext_only_w = sum(alpha_weights.get(s, 0) for s in ext_only_s)
            ext_only_max = max((alpha_weights.get(s, 0) for s in ext_only_s), default=0)
            wf_analysis = {
                "ext_recovery_pct": 100.0, "cipsi_recovery_pct": 100.0*cipsi_weight/max(total_weight, 1e-20),
                "ext_per_config": total_weight/max(len(ext_set_a), 1),
                "cipsi_per_config": cipsi_weight/max(len(cipsi_set_a), 1),
                "categories": {"ext_only": {"n": len(ext_only_s), "pct": 100.0*ext_only_w/max(total_weight, 1e-20), "max": ext_only_max}},
            }
            if wf_analysis["ext_per_config"] > 0:
                wf_analysis["efficiency_ratio"] = wf_analysis["cipsi_per_config"]/wf_analysis["ext_per_config"]
            avg_ext_only = ext_only_w/max(len(ext_only_s), 1)
            if avg_ext_only > 0:
                wf_analysis["selectivity_ratio"] = wf_analysis["cipsi_per_config"]/avg_ext_only

            def cum_curve(config_set, weights, max_pts=100):
                ws = sorted([weights.get(s, 0) for s in config_set], reverse=True)
                cum = np.cumsum(ws)
                if len(cum) == 0: return [], []
                cum_pct = (cum/max(total_weight, 1e-20)*100).tolist()
                if len(cum_pct) <= max_pts: return list(range(1, len(cum_pct)+1)), cum_pct
                step = max(1, len(cum_pct)//max_pts)
                idx = list(range(0, len(cum_pct), step))
                if idx[-1] != len(cum_pct)-1: idx.append(len(cum_pct)-1)
                return [i+1 for i in idx], [cum_pct[i] for i in idx]
            full_set_a = set(int(s) for s in strsa)
            wf_analysis["opt_curve"] = dict(zip(["x","y"], cum_curve(full_set_a, alpha_weights)))
            wf_analysis["ext_curve"] = dict(zip(["x","y"], cum_curve(ext_set_a, alpha_weights)))
            wf_analysis["cipsi_curve"] = dict(zip(["x","y"], cum_curve(cipsi_set_a, alpha_weights)))

    # === State-resolved α/β marginals ===
    wf_amps = {}
    try:
        for lb in TARGET_STATES:
            if lb not in fci_labeled: continue
            entry = {}
            fi = fci_idx_map.get(lb)
            if fi is not None and fi < len(c_fci):
                c_r = np.asarray(c_fci[fi], float)
                if c_r.ndim == 2:
                    pa_r, pb_r = np.sum(c_r**2, axis=1), np.sum(c_r**2, axis=0)
                    if len(pa_r) == n_alpha_full and len(pb_r) == n_beta_full:
                        entry["ref_a"], entry["ref_b"] = pa_r.tolist(), pb_r.tolist()
                    else:
                        pa_full, pb_full = np.zeros(n_alpha_full), np.zeros(n_beta_full)
                        pa_full[:len(pa_r)] = pa_r; pb_full[:len(pb_r)] = pb_r
                        entry["ref_a"], entry["ref_b"] = pa_full.tolist(), pb_full.tolist()
            ei = ext_idx_map.get(lb)
            if ei is not None and ei < len(cl_ext):
                pa_e, pb_e = compute_state_marginals(cl_ext[ei], ext_a, ext_b, full_a_map, full_b_map, n_alpha_full, n_beta_full)
                if pa_e is not None: entry["ext_a"], entry["ext_b"] = pa_e.tolist(), pb_e.tolist()
            ci_i = cipsi_idx_map.get(lb)
            if ci_i is not None and ci_i < len(cl_cipsi):
                pa_c, pb_c = compute_state_marginals(cl_cipsi[ci_i], basis_a, basis_b, full_a_map, full_b_map, n_alpha_full, n_beta_full)
                if pa_c is not None: entry["cipsi_a"], entry["cipsi_b"] = pa_c.tolist(), pb_c.tolist()
            if entry: wf_amps[lb] = entry
        if wf_amps:
            print(f"\n    [AMP] States: {list(wf_amps.keys())}, α:{n_alpha_full}, β:{n_beta_full}")
    except Exception as e:
        print(f"\n    [AMP] Failed: {e}")

    wf_amps_meta = {"n_alpha_full": n_alpha_full, "n_beta_full": n_beta_full, "states_stored": list(wf_amps.keys())}

    # Important missing
    n_important, max_missing_w = 0, 0.0
    if cr_ext is not None and cr_ext.ndim == 2:
        for s in missing_a:
            ia = ext_str_map_a.get(s)
            if ia is not None and ia < cr_ext.shape[0]:
                w = float(np.max(np.abs(cr_ext[ia, :])))
                max_missing_w = max(max_missing_w, w)
                if w > 0.01: n_important += 1
    diag["n_important_missing"] = n_important; diag["max_missing_weight"] = max_missing_w

    if len(missing_a) > 0:
        print(f"\n    [DIAG] Missing alpha: {len(missing_a)}/{len(ext_set_a)} (important: {n_important})")
    conv_str = " → ".join(f"({c['D_a']},{c['D_b']})" for c in cipsi_convergence)
    print(f"\n    [CONV] {conv_str}")
    if wf_analysis:
        eo = wf_analysis.get("categories", {}).get("ext_only", {})
        print(f"    [WF] Ext: {wf_analysis['ext_recovery_pct']:.2f}% | CIPSI: {wf_analysis['cipsi_recovery_pct']:.2f}% | "
              f"eff: {wf_analysis.get('efficiency_ratio', 0):.1f}x | ext-only: {eo.get('n', 0)} ({eo.get('pct', 0):.4f}%)")

    dim_ext = max(len(ext_a), len(ext_b))
    dim_cipsi = max(len(basis_a), len(basis_b))
    print(f"  Ref:{dt_fci:.0f}s  SQD({len(sqd_a)},{len(sqd_b)}):{dt_sqd:.0f}s  "
          f"Ext({len(ext_a)},{len(ext_b)}):{dt_ext:.0f}s  CIPSI({len(basis_a)},{len(basis_b)}):{dt_cipsi:.0f}s  "
          f"[{time.time()-t0_all:.0f}s]", flush=True)

    return {
        "R": R, "fci": fci_labeled, "ext_sqd": ext_labeled, "cipsi": cipsi_labeled,
        "fci_s2": fci_s2, "ext_s2": ext_s2, "cipsi_s2": cipsi_s2,
        "dims": {"sqd_a": len(sqd_a), "sqd_b": len(sqd_b),
                 "ext_a": len(ext_a), "ext_b": len(ext_b), "ext": dim_ext,
                 "cipsi_a": len(basis_a), "cipsi_b": len(basis_b), "cipsi": dim_cipsi,
                 "shots": N_SHOTS*N_SEEDS},
        "diag": diag, "cipsi_convergence": cipsi_convergence, "wf_analysis": wf_analysis,
        "nat_occ": nat_occ, "wf_amps": wf_amps, "wf_amps_meta": wf_amps_meta,
    }


# %% Cell 6: Run PES scan (incremental save)

R_VALUES = [1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 
            1.40, 1.45, 1.50, 1.60, 1.70, 1.80, 1.90, 2.00,
            2.10, 2.20, 2.30, 2.40, 2.50, 2.60, 2.70, 2.80,
            2.90, 3.00]
REF_METHOD = "fci"
REF_NROOTS = 20

out_fname = f"cn_{NORB}o_pes.json"

print(f"CN radical (9e,{NORB}o)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE version")
print(f"R = {R_VALUES}  ({len(R_VALUES)} points)")
print(f"Ref = {REF_METHOD} ({REF_NROOTS} roots)")
print(f"Output: {out_fname}")

strsa_full, strsb_full = det_strings(NORB, NELEC)
print(f"Full space: {len(strsa_full)} alpha x {len(strsb_full)} beta = {len(strsa_full)*len(strsb_full):,} det")

t_total = time.time()
all_data = []

for R in R_VALUES:
    pt = compute_point(R, REF_NROOTS)
    all_data.append(pt)
    try:
        tmp_fname = out_fname + ".tmp"
        with open(tmp_fname, "w") as f:
            json.dump(all_data, f, indent=2)
        os.replace(tmp_fname, out_fname)
        print(f"    [SAVED] {out_fname}  ({len(all_data)}/{len(R_VALUES)} pts)", flush=True)
    except Exception as e:
        print(f"    [SAVE FAILED] {e}", flush=True)

print(f"\nTotal PES time: {time.time()-t_total:.0f}s")
print(f"[final saved] {out_fname}  ({len(all_data)} points)")


# %% Cell 7: Summary

all_labels = set()
for pt in all_data:
    all_labels |= set(pt["fci"].keys())
STATES_FOUND = [lb for lb in TARGET_STATES if lb in all_labels]
print(f"States: {STATES_FOUND}")

print(f"\n{'R':>5} |", end="")
for lb in STATES_FOUND:
    print(f" {lb+' FCI':>12} {lb+' Ext':>10} {lb+' CIPSI':>10} |", end="")
print()
print("-" * (7 + len(STATES_FOUND) * 35))

for pt in all_data:
    R = pt["R"]; row = f"{R:5.2f} |"
    for lb in STATES_FOUND:
        ef = pt["fci"].get(lb); ee = pt["ext_sqd"].get(lb); eh = pt["cipsi"].get(lb)
        row += f" {ef:12.6f}" if ef else f" {'':>12}"
        row += f" {(ee-ef)*1000:+9.2f}" if ee and ef else f" {'MISS':>10}"
        row += f" {(eh-ef)*1000:+9.2f}" if eh and ef else f" {'MISS':>10}"
        row += " |"
    print(row)

print("\nDone.")

PySCF threads: 24
Numba JIT: enabled (4 functions compiled on first call)
CN radical (9e,16o)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE version
R = [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.9, 1.95, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]  (34 points)
Ref = fci (20 roots)
Output: cn_16o_pes.json
Full space: 4368 alpha x 1820 beta = 7,949,760 det

  R = 0.80 A
    [BROAD] D=(1529,1504)
      D0: ΔE=+0.5938 mHa
      D1: ΔE=+1.0550 mHa
      D2: ΔE=+0.9834 mHa
      Q1: ΔE=+3.2624 mHa
    [EN] Broad D=(1529,1504) → Final D=(2033,1535)
      D0: broad=+0.5938 → final=+0.0745 mHa
      D1: broad=+1.0550 → final=+0.1256 mHa
      D2: broad=+0.9834 → final=+0.1170 mHa
      Q1: broad=+3.2624 → final=+0.1900 mHa

    [AMP] States: ['D0', 'D1', 'D2', 'Q1'], α:4368, β:1820

    [DIAG] Missing alpha: 2128/4010 (important: 0)

    [CONV] (1529,1504) → (1980,1529) → (2033,1535) → (2033,1535) → (2033,1535)
